In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window

# IMPORT CONFIG (from your GitHub-synced config file) #
from config import FAERS_FILE_PATHS, BRONZE_BASE_PATH, dedup_keys

# INITIALIZING SPARK SESSION #
spark = SparkSession.builder.appName("FAERS_PHARMA_ETL_PIPELINE").getOrCreate()

#---------------------------------------------------------------------------------------------------#
# PHASE 1 — EXTRACTION & RAW INGESTION #
# DATA READING #

# Reading FAERS files for 25Q4 #
dataframes = {}
for name, file in FAERS_FILE_PATHS.items():
    path = f"/Volumes/workspace/faers/faers_files/{file}"
    dataframes[name] = spark.read.csv(
        path,
        header=True,
        sep="$",
        inferSchema=True
    )

# Reading CT files for 25Q4 #
df_ct = spark.read.csv(
    "/Volumes/workspace/ct/ct_files/clinical_trials.csv",
    header=True,
    sep=",",
    inferSchema=True
)

#---------------------------------------------------------------------------------------------------#
# PHASE 2 — RAW PARQUET CONVERSION #

# SAVING FAERS FILES AS PARQUET #
for name, df in dataframes.items():
    bronze_raw_parquet_path = f"{BRONZE_BASE_PATH}/faers_parquet/raw/{name}"
    df.write.mode("overwrite").parquet(bronze_raw_parquet_path)
    print(f"{name.upper()} parquet file saved successfully")

# SAVING CT FILE AS PARQUET #
df_ct.write.mode("overwrite").parquet(
    f"{BRONZE_BASE_PATH}/ct_parquet/raw/ct"
)
print("CT parquet file saved successfully")

#---------------------------------------------------------------------------------------------------#
# PHASE 3 — BRONZE LAYER CLEANING & STANDARDIZATION #

# FAERS File #

faers_raw_paths = {
    name: f"{BRONZE_BASE_PATH}/faers_parquet/raw/{name}"
    for name in FAERS_FILE_PATHS.keys()
}

for name, path in faers_raw_paths.items():
    # Read Raw Data
    df_raw_faers = spark.read.parquet(path)
    # Deduplicated Records
    df_clean_faers = df_raw_faers.dropDuplicates(dedup_keys[name])
    # Dropping Records for Null PrimaryId
    df_clean_faers = df_clean_faers.filter(F.col("primaryid").isNotNull())
    # Typecasted Date Columns
    for col_name in df_clean_faers.columns:

        if col_name.endswith("_dt"):

            date_value = F.trim(F.col(col_name).cast("long").cast("string"))

            df_clean_faers = df_clean_faers.withColumn(

            col_name,

            F.when(
                date_value.rlike(r"^\d{8}$"),
                F.to_date(date_value, "yyyyMMdd")
            )

            .when(
                date_value.rlike(r"^\d{6}$"),
                F.to_date(
                    F.concat(date_value, F.lit("01")),
                    "yyyyMMdd"
                )
            )

            .when(
                date_value.rlike(r"^\d{4}$"),
                F.to_date(
                    F.concat(date_value, F.lit("0101")),
                    "yyyyMMdd"
                )
            )

            .otherwise(
                F.lit(None).cast("date")
            )
        )

    # SAVE CLEANED FAERS BRONZE DATA #

    bronze_cleaned_parquet_path = f"{BRONZE_BASE_PATH}/faers_parquet/cleaned/{name}"
    df_clean_faers.write.mode("overwrite").parquet(bronze_cleaned_parquet_path)
    print(f"{name.upper()} cleaned parquet file saved successfully")

# CT File #

ct_raw_path = f"{BRONZE_BASE_PATH}/ct_parquet/raw/ct"
# Read Raw Data
df_raw_ct = spark.read.parquet(ct_raw_path)
# Deduplicated Records
df_clean_ct = df_raw_ct.dropDuplicates(["nct_id"])
# Typecasted Date Columns
for col_name in df_clean_ct.columns:
    if col_name.endswith("_date"):
        date_value = F.trim(F.col(col_name).cast("string"))

        df_clean_ct = df_clean_ct.withColumn(
            col_name,
            F.when(
                date_value.rlike(r"^\d{4}-\d{2}-\d{2}$"),
                F.to_date(date_value, "yyyy-MM-dd")
            ).when(
                date_value.rlike(r"^\d{4}-\d{2}$"),
                F.to_date(
                    F.concat(date_value, F.lit("-01")),
                    "yyyy-MM-dd"
                )
            ).when(
                date_value.rlike(r"^\d{8}$"),
                F.to_date(date_value, "yyyyMMdd")
            ).otherwise(
                F.lit(None).cast("date")
            )
        )

# SAVE CLEANED CT BRONZE DATA #

df_clean_ct.write.mode("overwrite").parquet(f"{BRONZE_BASE_PATH}/ct_parquet/cleaned/ct")
print("CT cleaned parquet file saved successfully")

#---------------------------------------------------------------------------------------------------#